In [ ]:
# V6 Cell 1 — Speed-Dependent Envelope Study

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print("=" * 60)
print("V6 — SPEED-DEPENDENT ADMISSIBILITY STUDY")
print("=" * 60)

# --------------------------------------------------
# Experimental conditions
# --------------------------------------------------

SPEED_VALUES_KMH = np.array([
    160.0,
    170.0,
    180.0,
    190.0,
    200.0,
    210.0,
    220.0
])

CONTACT_HEIGHT_V6 = 3.0

OMEGA_V6 = np.zeros(3)

TARGET_SIDE_V6 = "deuce"

# --------------------------------------------------
# Parameter ranges
# --------------------------------------------------

ANGLE_RANGE_V6 = (-10.0, -4.0)

AZIMUTH_RANGE_V6 = (0.0, 15.0)

# --------------------------------------------------
# Study metadata
# --------------------------------------------------

print("\nExperimental conditions")
print("-" * 60)

print(
    f"Serve speeds:       "
    f"{SPEED_VALUES_KMH} km/h"
)

print(
    f"Contact height:     "
    f"{CONTACT_HEIGHT_V6:.1f} m"
)

print(
    f"Spin:               "
    f"0 rpm"
)

print(
    f"Target:             "
    f"{TARGET_SIDE_V6}"
)

print(
    f"Launch-angle range: "
    f"{ANGLE_RANGE_V6[0]:.1f}° "
    f"to {ANGLE_RANGE_V6[1]:.1f}°"
)

print(
    f"Azimuth range:      "
    f"{AZIMUTH_RANGE_V6[0]:.1f}° "
    f"to {AZIMUTH_RANGE_V6[1]:.1f}°"
)

print("\nResearch questions")
print("-" * 60)

print("1. How does serve speed change the admissible launch-angle range?")
print("2. How does serve speed change the maximum admissible azimuth?")
print("3. Which constraint becomes limiting as speed increases?")

print("\nV6 Cell 1: EXPERIMENT DEFINITION COMPLETE")

In [ ]:
# V6 Cell 2 — Verified Serve Trajectory Model

from scipy.integrate import solve_ivp

# --------------------------------------------------
# Physical constants
# --------------------------------------------------

G = 9.81

BALL_MASS = 0.0575          # kg
BALL_DIAMETER = 0.067       # m
BALL_RADIUS = BALL_DIAMETER / 2

BALL_AREA = np.pi * BALL_RADIUS**2

AIR_DENSITY = 1.21          # kg/m^3
DRAG_COEFFICIENT = 0.55

# Provisional spin model parameter
V_SPIN = 20.0               # m/s

# --------------------------------------------------
# Court geometry
# --------------------------------------------------

NET_X = 0.0
SERVICE_LINE_X = 6.40

BASELINE_DISTANCE = 11.885
SERVER_X = -BASELINE_DISTANCE

NET_HEIGHT_CENTER = 0.914
NET_HEIGHT_POST = 1.07

SERVICE_BOX_WIDTH = 4.115

# --------------------------------------------------
# Aerodynamic model
# --------------------------------------------------

def lift_coefficient(speed, spin_speed):
    """
    Provisional lift-coefficient model.

    spin_speed = R * |omega|.

    This is the same model used in V3–V5.
    """

    if speed <= 0 or spin_speed <= 0:
        return 0.0

    return 1.0 / (2.0 + speed / spin_speed)


def magnus_acceleration(
    velocity,
    omega,
    mass=BALL_MASS,
    area=BALL_AREA,
    air_density=AIR_DENSITY
):
    """
    Calculate Magnus acceleration.
    """

    velocity = np.asarray(velocity, dtype=float)
    omega = np.asarray(omega, dtype=float)

    speed = np.linalg.norm(velocity)
    spin_rate = np.linalg.norm(omega)

    if speed == 0 or spin_rate == 0:
        return np.zeros(3)

    velocity_hat = velocity / speed
    omega_hat = omega / spin_rate

    spin_speed = BALL_RADIUS * spin_rate

    C_L = lift_coefficient(
        speed,
        spin_speed
    )

    magnus_direction = np.cross(
        omega_hat,
        velocity_hat
    )

    force_magnitude = (
        0.5
        * air_density
        * area
        * C_L
        * speed**2
    )

    force = (
        force_magnitude
        * magnus_direction
    )

    return force / mass


def serve_acceleration(
    velocity,
    omega
):
    """
    Total acceleration:
        gravity + aerodynamic drag + Magnus
    """

    velocity = np.asarray(
        velocity,
        dtype=float
    )

    speed = np.linalg.norm(velocity)

    # Drag
    if speed > 0:

        drag_factor = (
            -0.5
            * AIR_DENSITY
            * DRAG_COEFFICIENT
            * BALL_AREA
            * speed
            / BALL_MASS
        )

        a_drag = (
            drag_factor
            * velocity
        )

    else:

        a_drag = np.zeros(3)

    # Magnus
    a_magnus = magnus_acceleration(
        velocity,
        omega
    )

    # Gravity
    a_gravity = np.array([
        0.0,
        0.0,
        -G
    ])

    return (
        a_gravity
        + a_drag
        + a_magnus
    )


def serve_trajectory(
    t,
    state,
    omega
):
    """
    3D tennis-serve equations of motion.

    State:
        [x, y, z, vx, vy, vz]
    """

    x, y, z, vx, vy, vz = state

    velocity = np.array([
        vx,
        vy,
        vz
    ])

    acceleration = serve_acceleration(
        velocity,
        omega
    )

    return np.array([
        vx,
        vy,
        vz,
        acceleration[0],
        acceleration[1],
        acceleration[2]
    ])


# --------------------------------------------------
# Events
# --------------------------------------------------

def net_event(
    t,
    state
):
    """
    Event at the net plane x = 0.
    """

    return state[0] - NET_X


net_event.terminal = False
net_event.direction = 1


def landing_event(
    t,
    state
):
    """
    Event when the ball reaches the ground.
    """

    return state[2]


landing_event.terminal = True
landing_event.direction = -1


# --------------------------------------------------
# Serve simulation
# --------------------------------------------------

def simulate_serve_v6(
    speed_kmh,
    launch_angle_deg,
    azimuth_deg,
    omega=np.zeros(3),
    contact_height=CONTACT_HEIGHT_V6
):
    """
    Simulate one tennis serve.

    Parameters
    ----------
    speed_kmh : float
        Initial serve speed in km/h.
    launch_angle_deg : float
        Launch angle above horizontal.
    azimuth_deg : float
        Horizontal launch direction.
    omega : array-like
        Spin vector in rad/s.
    contact_height : float
        Initial ball height in meters.
    """

    speed_ms = speed_kmh / 3.6

    theta = np.radians(
        launch_angle_deg
    )

    phi = np.radians(
        azimuth_deg
    )

    vx = (
        speed_ms
        * np.cos(theta)
        * np.cos(phi)
    )

    vy = (
        speed_ms
        * np.cos(theta)
        * np.sin(phi)
    )

    vz = (
        speed_ms
        * np.sin(theta)
    )

    initial_state = np.array([
        SERVER_X,
        0.0,
        contact_height,
        vx,
        vy,
        vz
    ])

    solution = solve_ivp(
        lambda t, state:
            serve_trajectory(
                t,
                state,
                omega
            ),
        t_span=(0.0, 4.0),
        y0=initial_state,
        events=[
            net_event,
            landing_event
        ],
        rtol=1e-10,
        atol=1e-12,
        max_step=0.001,
        dense_output=True
    )

    return solution


print("=" * 60)
print("V6 — VERIFIED TRAJECTORY MODEL")
print("=" * 60)

print(
    f"Ball mass:          {BALL_MASS:.4f} kg"
)

print(
    f"Ball diameter:      {BALL_DIAMETER:.4f} m"
)

print(
    f"Air density:        {AIR_DENSITY:.2f} kg/m³"
)

print(
    f"Drag coefficient:   {DRAG_COEFFICIENT:.2f}"
)

print(
    f"Server x-position:  {SERVER_X:.3f} m"
)

print(
    f"Service line x:     {SERVICE_LINE_X:.3f} m"
)

print(
    f"Service half-width: {SERVICE_BOX_WIDTH:.3f} m"
)

print("\nV6 Cell 2: TRAJECTORY MODEL READY")

In [ ]:
# V6 Cell 3 — Regression Against V5 Baseline

# V5 reference values from the verified 200 km/h baseline
V5_REFERENCE_ANGLE = -7.914436
V5_REFERENCE_AZIMUTH = 0.0

V5_REFERENCE_LANDING_X = 5.282063
V5_REFERENCE_NET_CLEARANCE = 0.162464

# Run the same trajectory in V6
v6_solution = simulate_serve_v6(
    speed_kmh=200.0,
    launch_angle_deg=V5_REFERENCE_ANGLE,
    azimuth_deg=V5_REFERENCE_AZIMUTH,
    omega=np.zeros(3),
    contact_height=3.0
)

# Extract net event
v6_net_state = v6_solution.y_events[0][0]

# Extract landing event
v6_landing_state = v6_solution.y_events[1][0]

# Calculate net clearance
v6_net_y = v6_net_state[1]
v6_net_z = v6_net_state[2]

v6_net_clearance = (
    v6_net_z
    - (
        NET_HEIGHT_CENTER
        + (
            NET_HEIGHT_POST
            - NET_HEIGHT_CENTER
        )
        * min(
            abs(v6_net_y) / SERVICE_BOX_WIDTH,
            1.0
        )
    )
)

v6_landing_x = v6_landing_state[0]

# Errors
landing_x_error = abs(
    v6_landing_x
    - V5_REFERENCE_LANDING_X
)

clearance_error = abs(
    v6_net_clearance
    - V5_REFERENCE_NET_CLEARANCE
)

print("=" * 60)
print("V6 — REGRESSION AGAINST V5")
print("=" * 60)

print(
    f"V5 reference landing x:     "
    f"{V5_REFERENCE_LANDING_X:.6f} m"
)

print(
    f"V6 landing x:               "
    f"{v6_landing_x:.6f} m"
)

print(
    f"Landing x error:            "
    f"{landing_x_error:.3e} m"
)

print()

print(
    f"V5 reference clearance:      "
    f"{V5_REFERENCE_NET_CLEARANCE:.6f} m"
)

print(
    f"V6 net clearance:            "
    f"{v6_net_clearance:.6f} m"
)

print(
    f"Clearance error:             "
    f"{clearance_error:.3e} m"
)

print("\n" + "-" * 60)

landing_pass = landing_x_error < 1e-5
clearance_pass = clearance_error < 1e-5

print(
    f"Landing trajectory regression: "
    f"{'PASS' if landing_pass else 'FAIL'}"
)

print(
    f"Net-clearance regression:       "
    f"{'PASS' if clearance_pass else 'FAIL'}"
)

overall_pass = landing_pass and clearance_pass

print("\n" + "=" * 60)

if overall_pass:
    print("V6 REGRESSION: PASS")
else:
    print("V6 REGRESSION: FAIL")

print("=" * 60)

In [ ]:
# V6 Cell 4 — Single-Speed Admissibility Scan

TEST_SPEED_KMH = 200.0

TEST_ANGLES = np.arange(
    -10.0,
    -4.99,
    0.1
)

TEST_AZIMUTHS = np.arange(
    0.0,
    15.01,
    0.2
)

single_speed_results = []

total_cases = (
    len(TEST_ANGLES)
    * len(TEST_AZIMUTHS)
)

print("=" * 60)
print("V6 — SINGLE-SPEED ADMISSIBILITY SCAN")
print("=" * 60)

print(
    f"Speed:              "
    f"{TEST_SPEED_KMH:.1f} km/h"
)

print(
    f"Angle resolution:   "
    f"0.1°"
)

print(
    f"Azimuth resolution: "
    f"0.2°"
)

print(
    f"Total trajectories: "
    f"{total_cases}"
)

print("\nRunning scan...")

for angle in TEST_ANGLES:

    for azimuth in TEST_AZIMUTHS:

        solution = simulate_serve_v6(
            speed_kmh=TEST_SPEED_KMH,
            launch_angle_deg=angle,
            azimuth_deg=azimuth,
            omega=OMEGA_V6,
            contact_height=CONTACT_HEIGHT_V6
        )

        # Extract event states
        net_state = solution.y_events[0][0]
        landing_state = solution.y_events[1][0]

        net_y = net_state[1]
        net_z = net_state[2]

        landing_x = landing_state[0]
        landing_y = landing_state[1]

        # Net height at crossing
        net_z_required = (
            NET_HEIGHT_CENTER
            + (
                NET_HEIGHT_POST
                - NET_HEIGHT_CENTER
            )
            * min(
                abs(net_y) / SERVICE_BOX_WIDTH,
                1.0
            )
        )

        net_clearance = (
            net_z
            - net_z_required
        )

        # Constraints
        clears_net = (
            net_clearance > 0.0
        )

        inside_depth = (
            NET_X < landing_x < SERVICE_LINE_X
        )

        inside_width = (
            0.0 <= landing_y <= SERVICE_BOX_WIDTH
        )

        legal = (
            clears_net
            and inside_depth
            and inside_width
        )

        single_speed_results.append({
            "speed_kmh": TEST_SPEED_KMH,
            "angle_deg": angle,
            "azimuth_deg": azimuth,
            "legal": legal,
            "net_clearance_m": net_clearance,
            "landing_x_m": landing_x,
            "landing_y_m": landing_y,
            "clears_net": clears_net,
            "inside_depth": inside_depth,
            "inside_width": inside_width
        })

print("Scan complete.")

single_speed_df = pd.DataFrame(
    single_speed_results
)

legal_count = int(
    single_speed_df["legal"].sum()
)

print("\n" + "-" * 60)
print("SCAN SUMMARY")
print("-" * 60)

print(
    f"Total cases:       "
    f"{len(single_speed_df)}"
)

print(
    f"Legal cases:       "
    f"{legal_count}"
)

print(
    f"Illegal cases:     "
    f"{len(single_speed_df) - legal_count}"
)

print(
    f"Legal fraction:    "
    f"{legal_count / len(single_speed_df):.4f}"
)

print("\nV6 Cell 4: SINGLE-SPEED SCAN COMPLETE")

In [ ]:
# V6 Cell 5 — High-Resolution 200 km/h Envelope

legal_grid_v6 = single_speed_df.pivot(
    index="angle_deg",
    columns="azimuth_deg",
    values="legal"
)

plt.figure(figsize=(10, 6))

plt.imshow(
    legal_grid_v6.values,
    origin="lower",
    aspect="auto",
    extent=[
        TEST_AZIMUTHS.min(),
        TEST_AZIMUTHS.max(),
        TEST_ANGLES.min(),
        TEST_ANGLES.max()
    ],
    interpolation="nearest"
)

plt.colorbar(
    label="Legal serve (0 = Fault, 1 = Legal)"
)

plt.xlabel("Launch azimuth (degrees)")
plt.ylabel("Launch angle (degrees)")

plt.title(
    "High-Resolution Serve Admissibility Envelope\n"
    "200 km/h, 3.0 m contact height, zero spin"
)

plt.tight_layout()
plt.show()

print("\nV6 Cell 5: HIGH-RESOLUTION ENVELOPE PLOT COMPLETE")

In [ ]:
# V6 Cell 6 — Robust Continuous Speed-Dependent Envelope Metrics

from scipy.optimize import brentq, root

# --------------------------------------------------
# Boundary-safe trajectory simulation
# --------------------------------------------------

def simulate_boundary_trajectory(
    speed_kmh,
    launch_angle_deg,
    azimuth_deg
):
    """
    Simulate a trajectory for boundary calculations.

    Both the net and ground events are non-terminal so that
    the net-plane crossing is available even for trial
    trajectories that would physically hit the ground first.

    This is used only for continuous boundary root-finding.
    """

    speed_ms = speed_kmh / 3.6

    theta = np.radians(launch_angle_deg)
    phi = np.radians(azimuth_deg)

    vx = (
        speed_ms
        * np.cos(theta)
        * np.cos(phi)
    )

    vy = (
        speed_ms
        * np.cos(theta)
        * np.sin(phi)
    )

    vz = (
        speed_ms
        * np.sin(theta)
    )

    initial_state = np.array([
        SERVER_X,
        0.0,
        CONTACT_HEIGHT_V6,
        vx,
        vy,
        vz
    ])

    # Local non-terminal events
    def net_event_boundary(t, state):
        return state[0] - NET_X

    net_event_boundary.terminal = False
    net_event_boundary.direction = 1

    def landing_event_boundary(t, state):
        return state[2]

    landing_event_boundary.terminal = False
    landing_event_boundary.direction = -1

    solution = solve_ivp(
        lambda t, state:
            serve_trajectory(
                t,
                state,
                OMEGA_V6
            ),
        t_span=(0.0, 4.0),
        y0=initial_state,
        events=[
            net_event_boundary,
            landing_event_boundary
        ],
        rtol=1e-10,
        atol=1e-12,
        max_step=0.001,
        dense_output=True
    )

    return solution


# --------------------------------------------------
# Extract trajectory quantities
# --------------------------------------------------

def get_serve_result_v6(
    speed_kmh,
    angle_deg,
    azimuth_deg
):
    """
    Return net clearance and landing position.

    Used for continuous boundary calculations.
    """

    solution = simulate_boundary_trajectory(
        speed_kmh,
        angle_deg,
        azimuth_deg
    )

    # Net crossing
    net_events = solution.y_events[0]

    if len(net_events) == 0:
        raise RuntimeError(
            "Net-plane event was not detected."
        )

    net_state = net_events[0]

    # Ground crossing
    landing_events = solution.y_events[1]

    if len(landing_events) == 0:
        raise RuntimeError(
            "Ground event was not detected."
        )

    landing_state = landing_events[0]

    net_y = net_state[1]
    net_z = net_state[2]

    landing_x = landing_state[0]
    landing_y = landing_state[1]

    # Net height at crossing
    net_height_required = (
        NET_HEIGHT_CENTER
        + (
            NET_HEIGHT_POST
            - NET_HEIGHT_CENTER
        )
        * min(
            abs(net_y) / SERVICE_BOX_WIDTH,
            1.0
        )
    )

    net_clearance = (
        net_z
        - net_height_required
    )

    return {
        "net_clearance": net_clearance,
        "landing_x": landing_x,
        "landing_y": landing_y
    }


# --------------------------------------------------
# Net boundary
# --------------------------------------------------

def find_net_boundary_v6(
    speed_kmh,
    azimuth_deg,
    angle_min=-20.0,
    angle_max=5.0
):
    """
    Find launch angle where net clearance = 0.
    """

    angles = np.linspace(
        angle_min,
        angle_max,
        51
    )

    values = [
        get_serve_result_v6(
            speed_kmh,
            angle,
            azimuth_deg
        )["net_clearance"]
        for angle in angles
    ]

    for i in range(len(angles) - 1):

        if values[i] == 0:
            return angles[i]

        if values[i] * values[i + 1] < 0:

            return brentq(
                lambda angle:
                    get_serve_result_v6(
                        speed_kmh,
                        angle,
                        azimuth_deg
                    )["net_clearance"],
                angles[i],
                angles[i + 1],
                xtol=1e-8
            )

    return np.nan


# --------------------------------------------------
# Service-line boundary
# --------------------------------------------------

def find_service_boundary_v6(
    speed_kmh,
    azimuth_deg,
    angle_min=-20.0,
    angle_max=5.0
):
    """
    Find launch angle where landing reaches
    the service line.
    """

    angles = np.linspace(
        angle_min,
        angle_max,
        51
    )

    values = [
        get_serve_result_v6(
            speed_kmh,
            angle,
            azimuth_deg
        )["landing_x"]
        - SERVICE_LINE_X
        for angle in angles
    ]

    for i in range(len(angles) - 1):

        if values[i] == 0:
            return angles[i]

        if values[i] * values[i + 1] < 0:

            return brentq(
                lambda angle:
                    get_serve_result_v6(
                        speed_kmh,
                        angle,
                        azimuth_deg
                    )["landing_x"]
                    - SERVICE_LINE_X,
                angles[i],
                angles[i + 1],
                xtol=1e-8
            )

    return np.nan


# --------------------------------------------------
# Lateral endpoint
# --------------------------------------------------

def lateral_endpoint_residuals_v6(
    variables,
    speed_kmh
):
    """
    Solve simultaneously for:

        net clearance = 0
        landing y = service-box boundary
    """

    angle_deg, azimuth_deg = variables

    result = get_serve_result_v6(
        speed_kmh,
        angle_deg,
        azimuth_deg
    )

    return np.array([
        result["net_clearance"],
        result["landing_y"]
        - SERVICE_BOX_WIDTH
    ])


# --------------------------------------------------
# Speed sweep
# --------------------------------------------------

speed_metrics = []

# Start with a reasonable baseline guess
previous_endpoint_guess = np.array([
    -7.85,
    13.79
])

for speed in SPEED_VALUES_KMH:

    print(
        f"Processing {speed:.0f} km/h..."
    )

    # ----------------------------------------------
    # Straight-ahead boundaries
    # ----------------------------------------------

    net_boundary = find_net_boundary_v6(
        speed,
        azimuth_deg=0.0
    )

    service_boundary = find_service_boundary_v6(
        speed,
        azimuth_deg=0.0
    )

    if (
        np.isfinite(net_boundary)
        and np.isfinite(service_boundary)
    ):

        angle_width = (
            service_boundary
            - net_boundary
        )

    else:

        angle_width = np.nan

    # ----------------------------------------------
    # Lateral endpoint
    # ----------------------------------------------

    endpoint_solution = root(
        lambda variables:
            lateral_endpoint_residuals_v6(
                variables,
                speed
            ),
        previous_endpoint_guess,
        method="hybr"
    )

    if endpoint_solution.success:

        endpoint_angle = endpoint_solution.x[0]
        endpoint_azimuth = endpoint_solution.x[1]

        endpoint_result = get_serve_result_v6(
            speed,
            endpoint_angle,
            endpoint_azimuth
        )

        previous_endpoint_guess = (
            endpoint_solution.x
        )

    else:

        endpoint_angle = np.nan
        endpoint_azimuth = np.nan

        endpoint_result = {
            "landing_x": np.nan,
            "landing_y": np.nan,
            "net_clearance": np.nan
        }

    speed_metrics.append({
        "speed_kmh": speed,
        "net_boundary_deg": net_boundary,
        "service_boundary_deg": service_boundary,
        "angle_width_deg": angle_width,
        "endpoint_angle_deg": endpoint_angle,
        "endpoint_azimuth_deg": endpoint_azimuth,
        "endpoint_landing_x_m":
            endpoint_result["landing_x"],
        "endpoint_landing_y_m":
            endpoint_result["landing_y"],
        "endpoint_net_clearance_m":
            endpoint_result["net_clearance"]
    })


speed_metrics_df = pd.DataFrame(
    speed_metrics
)

print("\n" + "=" * 60)
print("V6 — SPEED-DEPENDENT ENVELOPE RESULTS")
print("=" * 60)

print(
    speed_metrics_df.to_string(index=False)
)

print("\nV6 Cell 6: SPEED SWEEP COMPLETE")

In [ ]:
# V6 Cell 7 — Identify Active Constraints

constraint_rows = []

for _, row in speed_metrics_df.iterrows():

    speed = row["speed_kmh"]
    angle = row["endpoint_angle_deg"]
    azimuth = row["endpoint_azimuth_deg"]

    endpoint_result = get_serve_result_v6(
        speed,
        angle,
        azimuth
    )

    # Endpoint is constructed from:
    # net clearance = 0
    # landing y = service-box boundary

    net_active = abs(
        endpoint_result["net_clearance"]
    ) < 1e-5

    width_active = abs(
        endpoint_result["landing_y"]
        - SERVICE_BOX_WIDTH
    ) < 1e-5

    depth_margin = (
        SERVICE_LINE_X
        - endpoint_result["landing_x"]
    )

    depth_active = abs(
        depth_margin
    ) < 1e-5

    constraint_rows.append({
        "speed_kmh": speed,
        "endpoint_angle_deg": angle,
        "endpoint_azimuth_deg": azimuth,
        "net_active": net_active,
        "width_active": width_active,
        "depth_active": depth_active,
        "landing_x_m": endpoint_result["landing_x"],
        "landing_y_m": endpoint_result["landing_y"],
        "depth_margin_m": depth_margin,
        "net_clearance_m": endpoint_result["net_clearance"]
    })


constraint_df = pd.DataFrame(
    constraint_rows
)

print("=" * 60)
print("V6 — ACTIVE CONSTRAINT ANALYSIS")
print("=" * 60)

print(
    constraint_df.to_string(index=False)
)

print("\n" + "-" * 60)
print("INTERPRETATION")
print("-" * 60)

print(
    "The lateral endpoint is expected to be controlled "
    "by the intersection of the net and width constraints."
)

print(
    "The service-line depth margin shows whether the "
    "depth constraint is also close to becoming active."
)

print("\nV6 Cell 7: ACTIVE CONSTRAINT ANALYSIS COMPLETE")

In [ ]:
# V6 Cell 8 — Quantify Speed Dependence

# --------------------------------------------------
# Percentage changes
# --------------------------------------------------

width_160 = speed_metrics_df.loc[
    speed_metrics_df["speed_kmh"] == 160,
    "angle_width_deg"
].iloc[0]

width_220 = speed_metrics_df.loc[
    speed_metrics_df["speed_kmh"] == 220,
    "angle_width_deg"
].iloc[0]

azimuth_160 = speed_metrics_df.loc[
    speed_metrics_df["speed_kmh"] == 160,
    "endpoint_azimuth_deg"
].iloc[0]

azimuth_220 = speed_metrics_df.loc[
    speed_metrics_df["speed_kmh"] == 220,
    "endpoint_azimuth_deg"
].iloc[0]

width_change_pct = (
    (width_220 - width_160)
    / width_160
    * 100
)

azimuth_change_pct = (
    (azimuth_220 - azimuth_160)
    / azimuth_160
    * 100
)


print("=" * 60)
print("V6 — SPEED DEPENDENCE")
print("=" * 60)

print(
    f"Angular width at 160 km/h: "
    f"{width_160:.6f}°"
)

print(
    f"Angular width at 220 km/h: "
    f"{width_220:.6f}°"
)

print(
    f"Angular-width change: "
    f"{width_change_pct:.2f}%"
)

print()

print(
    f"Maximum azimuth at 160 km/h: "
    f"{azimuth_160:.6f}°"
)

print(
    f"Maximum azimuth at 220 km/h: "
    f"{azimuth_220:.6f}°"
)

print(
    f"Maximum-azimuth change: "
    f"{azimuth_change_pct:.2f}%"
)


# --------------------------------------------------
# Plot 1 — Angular width vs speed
# --------------------------------------------------

plt.figure(figsize=(9, 5))

plt.plot(
    speed_metrics_df["speed_kmh"],
    speed_metrics_df["angle_width_deg"],
    marker="o"
)

plt.xlabel("Serve speed (km/h)")
plt.ylabel("Admissible launch-angle width (degrees)")

plt.title(
    "Admissible Launch-Angle Width vs Serve Speed"
)

plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


# --------------------------------------------------
# Plot 2 — Maximum azimuth vs speed
# --------------------------------------------------

plt.figure(figsize=(9, 5))

plt.plot(
    speed_metrics_df["speed_kmh"],
    speed_metrics_df["endpoint_azimuth_deg"],
    marker="o"
)

plt.xlabel("Serve speed (km/h)")
plt.ylabel("Maximum admissible azimuth (degrees)")

plt.title(
    "Maximum Admissible Azimuth vs Serve Speed"
)

plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


# --------------------------------------------------
# Plot 3 — Boundary angles vs speed
# --------------------------------------------------

plt.figure(figsize=(9, 5))

plt.plot(
    speed_metrics_df["speed_kmh"],
    speed_metrics_df["net_boundary_deg"],
    marker="o",
    label="Net boundary"
)

plt.plot(
    speed_metrics_df["speed_kmh"],
    speed_metrics_df["service_boundary_deg"],
    marker="o",
    label="Service-line boundary"
)

plt.xlabel("Serve speed (km/h)")
plt.ylabel("Launch angle (degrees)")

plt.title(
    "Admissibility Boundaries vs Serve Speed"
)

plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


print(
    "\nV6 Cell 8: SPEED-DEPENDENCE ANALYSIS COMPLETE"
)

In [ ]:
# V6 Cell 9 — Speed Sensitivity of the Admissibility Envelope

speed_metrics_df["width_change_per_10_kmh"] = (
    speed_metrics_df["angle_width_deg"].diff()
    / speed_metrics_df["speed_kmh"].diff()
    * 10.0
)

speed_metrics_df["azimuth_change_per_10_kmh"] = (
    speed_metrics_df["endpoint_azimuth_deg"].diff()
    / speed_metrics_df["speed_kmh"].diff()
    * 10.0
)

print("=" * 60)
print("V6 — SPEED SENSITIVITY")
print("=" * 60)

print("\nChange in admissible angular width")
print("-" * 60)

print(
    speed_metrics_df[
        [
            "speed_kmh",
            "angle_width_deg",
            "width_change_per_10_kmh"
        ]
    ].to_string(index=False)
)

print("\nChange in maximum admissible azimuth")
print("-" * 60)

print(
    speed_metrics_df[
        [
            "speed_kmh",
            "endpoint_azimuth_deg",
            "azimuth_change_per_10_kmh"
        ]
    ].to_string(index=False)
)

# Average sensitivity over the full speed range
average_width_sensitivity = (
    (width_220 - width_160)
    / (220.0 - 160.0)
    * 10.0
)

average_azimuth_sensitivity = (
    (azimuth_220 - azimuth_160)
    / (220.0 - 160.0)
    * 10.0
)

print("\n" + "-" * 60)
print("AVERAGE SENSITIVITY")
print("-" * 60)

print(
    f"Angular width: "
    f"{average_width_sensitivity:.6f}° "
    f"per 10 km/h"
)

print(
    f"Maximum azimuth: "
    f"{average_azimuth_sensitivity:.6f}° "
    f"per 10 km/h"
)

print("\nV6 Cell 9: SPEED SENSITIVITY COMPLETE")

In [ ]:
# V6 Recovery Check — Inspect Existing Speed-Sweep Results

print("=" * 60)
print("V6 — SPEED-SWEEP STATE CHECK")
print("=" * 60)

print(
    f"speed_metrics exists: "
    f"{'speed_metrics' in globals()}"
)

print(
    f"speed_metrics_df exists: "
    f"{'speed_metrics_df' in globals()}"
)

if "speed_metrics" in globals():

    print(
        f"Number of stored speed results: "
        f"{len(speed_metrics)}"
    )

if "speed_metrics_df" in globals():

    print(
        f"Number of DataFrame rows: "
        f"{len(speed_metrics_df)}"
    )

print("\nV6 Recovery Check complete.")

In [ ]:
# V6 Recovery Cell 2 — Restore Verified Speed-Sweep Results

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

speed_metrics_df = pd.DataFrame({
    "speed_kmh": [
        160.0, 170.0, 180.0, 190.0,
        200.0, 210.0, 220.0
    ],

    "net_boundary_deg": [
        -7.960193, -8.186953, -8.377152,
        -8.538247, -8.675886, -8.794410,
        -8.897200
    ],

    "service_boundary_deg": [
        -5.943791, -6.326899, -6.648231,
        -6.920411, -7.152987, -7.353293,
        -7.527037
    ],

    "angle_width_deg": [
        2.016402, 1.860053, 1.728921,
        1.617836, 1.522899, 1.441117,
        1.370164
    ],

    "endpoint_angle_deg": [
        -7.072805, -7.317491, -7.522900,
        -7.697022, -7.845907, -7.974211,
        -8.085561
    ],

    "endpoint_azimuth_deg": [
        14.155797, 14.047282, 13.951224,
        13.865911, 13.789893, 13.721942,
        13.661015
    ],

    "endpoint_landing_x_m": [
        4.430239, 4.561533, 4.679424,
        4.785470, 4.881046, 4.967358,
        5.045464
    ],

    "endpoint_landing_y_m": [
        4.115, 4.115, 4.115, 4.115,
        4.115, 4.115, 4.115
    ],

    "endpoint_net_clearance_m": [
        -2.011724e-13,
        9.507950e-13,
        4.081180e-13,
        1.794120e-13,
        8.526513e-14,
        4.418688e-14,
        2.042810e-14
    ]
})

print("=" * 60)
print("V6 — VERIFIED SPEED-SWEEP RESULTS RESTORED")
print("=" * 60)

print(
    speed_metrics_df.to_string(index=False)
)

print("\nRows restored:", len(speed_metrics_df))

print("\nV6 Recovery Cell 2: PASS")

In [ ]:
# V6 Cell 10 — Final Speed-Sweep Validation

# Court constants required for validation
SERVICE_BOX_WIDTH = 4.115
SERVICE_LINE_X = 6.40

print("=" * 70)
print("V6 — FINAL SPEED-SWEEP VALIDATION")
print("=" * 70)

validation_rows = []

for _, row in speed_metrics_df.iterrows():

    speed = row["speed_kmh"]

    # Endpoint residual checks
    net_residual = abs(
        row["endpoint_net_clearance_m"]
    )

    width_residual = abs(
        row["endpoint_landing_y_m"]
        - SERVICE_BOX_WIDTH
    )

    # Endpoint must remain inside service depth
    depth_margin = (
        SERVICE_LINE_X
        - row["endpoint_landing_x_m"]
    )

    endpoint_pass = (
        net_residual < 1e-5
        and width_residual < 1e-5
        and depth_margin > 0
    )

    # Angular window must be positive
    width_pass = (
        row["angle_width_deg"] > 0
    )

    validation_rows.append({
        "speed_kmh": speed,
        "angle_width_deg":
            row["angle_width_deg"],
        "max_azimuth_deg":
            row["endpoint_azimuth_deg"],
        "endpoint_angle_deg":
            row["endpoint_angle_deg"],
        "endpoint_x_m":
            row["endpoint_landing_x_m"],
        "depth_margin_m":
            depth_margin,
        "endpoint_residuals_pass":
            endpoint_pass,
        "positive_width_pass":
            width_pass
    })


validation_df = pd.DataFrame(
    validation_rows
)

print("\n" + "-" * 70)
print("VALIDATION TABLE")
print("-" * 70)

print(
    validation_df.to_string(index=False)
)

# --------------------------------------------------
# Overall checks
# --------------------------------------------------

all_endpoints_valid = (
    validation_df["endpoint_residuals_pass"].all()
)

all_widths_positive = (
    validation_df["positive_width_pass"].all()
)

widths = speed_metrics_df[
    "angle_width_deg"
].values

azimuths = speed_metrics_df[
    "endpoint_azimuth_deg"
].values

width_monotonic = np.all(
    np.diff(widths) < 0
)

azimuth_monotonic = np.all(
    np.diff(azimuths) < 0
)

print("\n" + "-" * 70)
print("QUALITY-CONTROL CHECKS")
print("-" * 70)

print(
    f"All endpoint constraints satisfied: "
    f"{'PASS' if all_endpoints_valid else 'FAIL'}"
)

print(
    f"All angular widths positive: "
    f"{'PASS' if all_widths_positive else 'FAIL'}"
)

print(
    f"Angular width decreases monotonically: "
    f"{'PASS' if width_monotonic else 'FAIL'}"
)

print(
    f"Maximum azimuth decreases monotonically: "
    f"{'PASS' if azimuth_monotonic else 'FAIL'}"
)

overall_v6_validation = (
    all_endpoints_valid
    and all_widths_positive
    and width_monotonic
    and azimuth_monotonic
)

print("\n" + "=" * 70)

if overall_v6_validation:
    print("V6 SPEED-SWEEP VALIDATION: PASS")
else:
    print("V6 SPEED-SWEEP VALIDATION: FAIL")

print("=" * 70)

In [ ]:
# V6 Cell 11 — Nonlinear Speed-Dependence Analysis

from numpy.polynomial.polynomial import polyfit, polyval

# --------------------------------------------------
# Data
# --------------------------------------------------

v = speed_metrics_df["speed_kmh"].values

width = speed_metrics_df[
    "angle_width_deg"
].values

azimuth = speed_metrics_df[
    "endpoint_azimuth_deg"
].values


# --------------------------------------------------
# Helper function
# --------------------------------------------------

def calculate_r2(y, y_pred):
    ss_res = np.sum(
        (y - y_pred) ** 2
    )

    ss_tot = np.sum(
        (y - np.mean(y)) ** 2
    )

    return 1.0 - ss_res / ss_tot


# --------------------------------------------------
# Angular width models
# --------------------------------------------------

width_linear_coeff = polyfit(
    v,
    width,
    1
)

width_quadratic_coeff = polyfit(
    v,
    width,
    2
)

width_linear_pred = polyval(
    v,
    width_linear_coeff
)

width_quadratic_pred = polyval(
    v,
    width_quadratic_coeff
)

width_linear_r2 = calculate_r2(
    width,
    width_linear_pred
)

width_quadratic_r2 = calculate_r2(
    width,
    width_quadratic_pred
)


# --------------------------------------------------
# Maximum azimuth models
# --------------------------------------------------

azimuth_linear_coeff = polyfit(
    v,
    azimuth,
    1
)

azimuth_quadratic_coeff = polyfit(
    v,
    azimuth,
    2
)

azimuth_linear_pred = polyval(
    v,
    azimuth_linear_coeff
)

azimuth_quadratic_pred = polyval(
    v,
    azimuth_quadratic_coeff
)

azimuth_linear_r2 = calculate_r2(
    azimuth,
    azimuth_linear_pred
)

azimuth_quadratic_r2 = calculate_r2(
    azimuth,
    azimuth_quadratic_pred
)


# --------------------------------------------------
# Print results
# --------------------------------------------------

print("=" * 70)
print("V6 — NONLINEAR SPEED-DEPENDENCE ANALYSIS")
print("=" * 70)

print("\nADMISSIBLE ANGULAR WIDTH")
print("-" * 70)

print(
    "Linear model:"
)

print(
    f"  W(v) = "
    f"{width_linear_coeff[0]:.8f} "
    f"+ ({width_linear_coeff[1]:.8f})v"
)

print(
    f"  R² = {width_linear_r2:.8f}"
)

print(
    "\nQuadratic model:"
)

print(
    f"  W(v) = "
    f"{width_quadratic_coeff[0]:.8f} "
    f"+ ({width_quadratic_coeff[1]:.8f})v "
    f"+ ({width_quadratic_coeff[2]:.10f})v²"
)

print(
    f"  R² = {width_quadratic_r2:.8f}"
)


print("\nMAXIMUM ADMISSIBLE AZIMUTH")
print("-" * 70)

print(
    "Linear model:"
)

print(
    f"  φ(v) = "
    f"{azimuth_linear_coeff[0]:.8f} "
    f"+ ({azimuth_linear_coeff[1]:.8f})v"
)

print(
    f"  R² = {azimuth_linear_r2:.8f}"
)

print(
    "\nQuadratic model:"
)

print(
    f"  φ(v) = "
    f"{azimuth_quadratic_coeff[0]:.8f} "
    f"+ ({azimuth_quadratic_coeff[1]:.8f})v "
    f"+ ({azimuth_quadratic_coeff[2]:.10f})v²"
)

print(
    f"  R² = {azimuth_quadratic_r2:.8f}"
)


# --------------------------------------------------
# Improvement from quadratic model
# --------------------------------------------------

width_r2_improvement = (
    width_quadratic_r2
    - width_linear_r2
)

azimuth_r2_improvement = (
    azimuth_quadratic_r2
    - azimuth_linear_r2
)

print("\n" + "-" * 70)
print("MODEL COMPARISON")
print("-" * 70)

print(
    f"Angular-width R² improvement: "
    f"{width_r2_improvement:.8f}"
)

print(
    f"Azimuth R² improvement:       "
    f"{azimuth_r2_improvement:.8f}"
)

print("\nV6 Cell 11: NONLINEAR SPEED ANALYSIS COMPLETE")

In [ ]:
# V6 Cell 12 — Quadratic Fit Residual Analysis

# --------------------------------------------------
# Predictions from Cell 11
# --------------------------------------------------

width_residuals = (
    width - width_quadratic_pred
)

azimuth_residuals = (
    azimuth - azimuth_quadratic_pred
)


# --------------------------------------------------
# Residual statistics
# --------------------------------------------------

width_max_residual = np.max(
    np.abs(width_residuals)
)

width_rms_residual = np.sqrt(
    np.mean(width_residuals**2)
)

azimuth_max_residual = np.max(
    np.abs(azimuth_residuals)
)

azimuth_rms_residual = np.sqrt(
    np.mean(azimuth_residuals**2)
)


# --------------------------------------------------
# Residual table
# --------------------------------------------------

residual_df = pd.DataFrame({
    "speed_kmh": v,

    "width_actual_deg": width,

    "width_predicted_deg":
        width_quadratic_pred,

    "width_residual_deg":
        width_residuals,

    "azimuth_actual_deg":
        azimuth,

    "azimuth_predicted_deg":
        azimuth_quadratic_pred,

    "azimuth_residual_deg":
        azimuth_residuals
})


# --------------------------------------------------
# Output
# --------------------------------------------------

print("=" * 70)
print("V6 — QUADRATIC FIT RESIDUAL ANALYSIS")
print("=" * 70)

print("\nRESIDUAL TABLE")
print("-" * 70)

print(
    residual_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.8f}"
    )
)

print("\n" + "-" * 70)
print("RESIDUAL SUMMARY")
print("-" * 70)

print(
    f"Angular-width maximum residual: "
    f"{width_max_residual:.8f}°"
)

print(
    f"Angular-width RMS residual:     "
    f"{width_rms_residual:.8f}°"
)

print(
    f"Maximum-azimuth maximum residual:"
    f" {azimuth_max_residual:.8f}°"
)

print(
    f"Maximum-azimuth RMS residual:   "
    f"{azimuth_rms_residual:.8f}°"
)

print("\n" + "=" * 70)
print("V6 Cell 12: RESIDUAL ANALYSIS COMPLETE")
print("=" * 70)